# N162 · JavaScript函数组合、缓存与事件

**目标：** 用函数对象组合行为并管理订阅生命周期。

**先修：** N161。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** map/filter/reduce；函数组合；memoization键；once；事件订阅/取消；对象引用。

## 从问题到算法

函数组合从右向左执行，因此 `compose([f,g])(x)=f(g(x))`；空组合保持输入。自定义 map 明确向回调传入“值和索引”，本章数组为稠密数组。

缓存键本身也是契约。把参数 `JSON.stringify` 后当键，会混淆某些值，并失去对象身份。本章用 `Map` 参数前缀树；参数个数由终止节点决定，返回 `undefined` 也有单独命中标记。函数必须是只依赖参数的纯函数；可变对象内容变化而身份不变时，不应期待缓存重新计算。

事件订阅用独立注册对象标识，同一个函数注册两次是两次订阅。emit 对当前订阅列表拍快照，本次触发期间的增删从下一次生效。

## 最小实现

**本章接口：** `map(arr, fn)`、`compose(functions)`、`memoize(fn)`、`EventEmitter`

In [1]:
JS_SOURCE = r''''use strict';
function map(arr, fn) {
  const out=[];
  for (let i=0;i<arr.length;i++) out.push(fn(arr[i],i));
  return out;
}
function compose(functions) {
  return function(x) {
    for (let i=functions.length-1;i>=0;i--) x=functions[i](x);
    return x;
  };
}
function memoize(fn) {
  // Contract: pure function of positional arguments; object keys use identity.
  const node=()=>({children:new Map(), ready:false, value:undefined});
  const root=node();
  return function(...args) {
    let current=root;
    for (const arg of args) {
      if (!current.children.has(arg)) current.children.set(arg,node());
      current=current.children.get(arg);
    }
    if (!current.ready) {
      current.value=fn(...args);
      current.ready=true;
    }
    return current.value;
  };
}
class EventEmitter {
  constructor() { this.events=new Map(); }
  subscribe(eventName,callback) {
    const item={callback};
    if (!this.events.has(eventName)) this.events.set(eventName,[]);
    this.events.get(eventName).push(item);
    let active=true;
    return {unsubscribe:()=>{
      if (!active) return;
      active=false;
      const items=this.events.get(eventName);
      const index=items.indexOf(item);
      if (index>=0) items.splice(index,1);
      if (items.length===0) this.events.delete(eventName);
    }};
  }
  emit(eventName,args=[]) {
    // Snapshot semantics: subscriptions changed during emission apply next time.
    const snapshot=[...(this.events.get(eventName)||[])];
    return snapshot.map(item=>item.callback(...args));
  }
}
'''

import json
import subprocess

def run_js(extra):
    result = subprocess.run(['node', '-e', JS_SOURCE + '\n' + extra],
                            text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 正确性、前提与复杂度

参数序列沿唯一前缀树路径到达缓存节点；终止标记区分尚未计算和计算结果为 undefined。事件快照保留订阅顺序，每个注册对象在一次 emit 中调用一次，取消操作只删除对应注册。

**代价：** map 和含 k 个函数的组合分别 O(n)、O(k)。a 个参数的缓存访问期望 O(a)，空间随不同参数前缀数增长。emit 对当前 m 个订阅 O(m)；数组式取消订阅 O(m)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from IPython.display import Code
display(Code(JS_SOURCE,language='javascript'))
trace = json.loads(run_js("const e=new EventEmitter(); const a=e.subscribe('x',v=>v+1); const b=e.subscribe('x',v=>v*2); const rows=[['two listeners',e.emit('x',[3])]]; a.unsubscribe(); rows.push(['after unsubscribe',e.emit('x',[3])]); console.log(JSON.stringify(rows));"))
show_table(['event state','results'],trace)

'use strict';
function map(arr, fn) {
  const out=[];
  for (let i=0;i<arr.length;i++) out.push(fn(arr[i],i));
  return out;
}
function compose(functions) {
  return function(x) {
    for (let i=functions.length-1;i>=0;i--) x=functions[i](x);
    return x;
  };
}
function memoize(fn) {
  // Contract: pure function of positional arguments; object keys use identity.
  const node=()=>({children:new Map(), ready:false, value:undefined});
  const root=node();
  return function(...args) {
    let current=root;
    for (const arg of args) {
      if (!current.children.has(arg)) current.children.set(arg,node());
      current=current.children.get(arg);
    }
    if (!current.ready) {
      current.value=fn(...args);
      current.ready=true;
    }
    return current.value;
  };
}
class EventEmitter {
  constructor() { this.events=new Map(); }
  subscribe(eventName,callback) {
    const item={callback};
    if (!this.events.has(eventName)) this.events.set(eventName,[]);
    this.events.get(eventName).push(item);
    let active=true;
    return {unsubscribe:()=>{
      if (!active) return;
      active=false;
      const items=this.events.get(eventName);
      const index=items.indexOf(item);
      if (index>=0) items.splice(index,1);
      if (items.length===0) this.events.delete(eventName);
    }};
  }
  emit(eventName,args=[]) {
    // Snapshot semantics: subscriptions changed during emission apply next time.
    const snapshot=[...(this.events.get(eventName)||[])];
    return snapshot.map(item=>item.callback(...args));
  }
}

event state,results
two listeners,"[4, 6]"
after unsubscribe,[6]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
print(run_js("""const assert=require('node:assert/strict');
assert.deepEqual(map([2,3],(x,i)=>x+i),[2,4]);
assert.deepEqual(map([],x=>x),[]); assert.equal(compose([])(7),7);
assert.equal(compose([x=>x+1,x=>x*2])(3),7);
let calls=0; const cached=memoize((...args)=>{calls++;return args.length;});
assert.equal(cached(1),1);assert.equal(cached(1),1);assert.equal(calls,1);
assert.equal(cached(1,undefined),2); assert.equal(cached('1'),1);
const obj={x:1};cached(obj);cached(obj);cached({x:1});assert.equal(calls,5);
let undefinedCalls=0;const missing=memoize(()=>{undefinedCalls++;return undefined;});
missing();missing();assert.equal(undefinedCalls,1);
const e=new EventEmitter();assert.deepEqual(e.emit('absent'),[]);
const cb=x=>x+1, a=e.subscribe('x',cb),b=e.subscribe('x',cb);
assert.deepEqual(e.emit('x',[2]),[3,3]);a.unsubscribe();a.unsubscribe();
assert.deepEqual(e.emit('x',[2]),[3]);b.unsubscribe();assert.deepEqual(e.emit('x'),[]);
const f=new EventEmitter();let second;
f.subscribe('t',()=>{second.unsubscribe();return 1;});
second=f.subscribe('t',()=>2);
assert.deepEqual(f.emit('t'),[1,2]);assert.deepEqual(f.emit('t'),[1]);
console.log('Node assertions passed');"""))
print('N162: 所有本章断言通过')

Node assertions passed

N162: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较按JSON文本键和对象身份缓存的不同契约。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 实现unsubscribe。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2635. Apply Transform Over Each Element in Array（canonical）
- 2629. Function Composition（canonical）
- 2623. Memoize（canonical）
- 2694. Event Emitter（canonical）：[官方入口](https://leetcode.com/problems/event-emitter/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。